In [1]:
# Computes stochastic control closure certificate using SOS for stochastic systems in verifying LTL specifications
# 3Tank Example

# include important libraries
using JuMP
using MosekTools
using DynamicPolynomials
using MultivariatePolynomials
using LinearAlgebra
using TSSOS # important for SOS, see https://github.com/wangjie212/TSSOS

@polyvar x1 x2 x3 x4 x5 x6 x7 x8 y1 y2 y3 y4 y5 y6 y7 y8 x01 x02 x03 x04 x05 x06 x07 x08 u1 u2 u3 u4 u5 # global vars used in monomials
var = [x1, x2, x3, x4, x5, x6, x7, x8, y1, y2, y3, y4, y5, y6, y7, y8]
sos_tol = 1 # the maximum degree of unknown SOS lagrange polynomials = deg + sos_tol 
error = 2   # precision digit places
tau, gamma, lamda, delta, tau1, tau2, tau3 = 2.0, 0.85, 125.0, 1.0, 0.2, 0.0025, 0.005 # SC3 constants and S-procedure constants

U = [0.0, 1.5, 4.5, 7.5, 9.0] 

rho = 1/length(U) # uniform control bilinearity coefficients
a, b1, b2, b = 0.0, 10.0, 60.0, 100.0 # state space
sig, mean = sqrt(0.01), 0 # Guassian noise standard deviation and mean
eps = 1e-6 # the lag for open intervals in the domain

# noise generator
function noise()
    return rand(Normal(mean, sig))
end

# build 3tank dynamics symbolically
function dyn(var1, dim)
    # 0.01*noise() working with linear C3 makes E[w]=0 for normal distribution
    x1n = (var1[4] - 0.5*tau)^2 
    x2n = (var1[6] - 0.5*tau)^2
    x3n = (var1[8] - 0.5*tau)^2
    if dim == 1
        return x1n
    elseif dim == 2
        return x2n
    else
        return x3n
    end
end
    
# access odd priorities
function odd_pri(d::Dict)
    return [v for v in values(d) if isodd(v)]
end

# creates the parametrized SCC
function add_paramC_poly!(model, var1, deg, q, p)
    basis = monomials(var1, 0:deg) # basis in variables
    coeffs = @variable(model, [1:length(basis)], base_name="c_$(q)_$(p)")
    C_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return C_qp, coeffs, basis
end

function add_paramCf_poly!(model, var1, deg, q, p)
    basis = monomials(var1, 0:deg) # basis in variables
    coeffs = @variable(model, [1:length(basis)], base_name="cf_$(q)_$(p)")
    Cf_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return Cf_qp, coeffs, basis
end

# creates the parametrized ranking functions
function add_paramV_poly!(model, var2, deg, l, q, p)
    basis = monomials(var2, 0:deg)
    coeffs = @variable(model, [1:length(basis)], base_name="v_$(l)_$(q)_$(p)")
    V_l_qp = sum(coeffs[i] * basis[i] for i in 1:length(basis))
    return V_l_qp, coeffs, basis
end

# the domain of dynamics for the SC3 constraints
# X0 = [a, 6]^2 x [60, 66]^2, X = [a, b]^3
g1a = [x1-a, 10-x1, x1-60, 100-x1, x2-a, 10-x2, x2-60, 100-x2, x3-a, 10-x3, x3-60, 100-x3]
g1b = [x1-10-eps, 60-eps-x1, x2-10-eps, 60-eps-x2, x3-10-eps, 60-eps-x3]
g = Dict(
:"a" => g1a, :"b" => g1b
    )
gX = [x1-a, b-x1, x2-a, b-x2, x3-a, b-x3]
gXY = [x1-a, b-x1, x2-a, b-x2, x3-a, b-x3, y1-a, b-y1, y2-a, b-y2, y3-a, b-y3]
gv = [x01-a, 6-x01, x02-a, 6-x02, x03-60, 66-x03, x1-a, b-x1, x2-a, b-x2, x3-a, b-x3, y1-a, b-y1, y2-a, b-y2, y3-a, b-y3]
var_5 = [x01, x02, x03, x04, x05, x06, x07, x08, x1, x2, x3, x4, x5, x6, x7, x8, y1, y2, y3, y4, y5, y6, y7, y8]

# other dynamics constraints
gu = [x4^2-((0.5*tau)^2+x1+tau*u1), -x4^2+((0.5*tau)^2+x1+tau*u1), x4^2-((0.5*tau)^2+x1+tau*u2), -x4^2+((0.5*tau)^2+x1+tau*u2),
x4^2-((0.5*tau)^2+x1+tau*u3), -x4^2+((0.5*tau)^2+x1+tau*u3), x4^2-((0.5*tau)^2+x1+tau*u4), -x4^2+((0.5*tau)^2+x1+tau*u4),
x4^2-((0.5*tau)^2+x1+tau*u5), -x4^2+((0.5*tau)^2+x1+tau*u5), u1-0.0, 0.0-u1, u2-1.5, 1.5-u2, u3-4.5, 4.5-u3, u4-7.5, 7.5-u4, u5-9.0, 9.0-u5]

g0 = [x6^2-((0.5*tau)^2+x2+tau*x5), -x6^2+((0.5*tau)^2+x2+tau*x5), 
    x8^2-((0.5*tau)^2+x3+tau*x7), -x8^2+((0.5*tau)^2+x3+tau*x7), x5^2-x1, -x5^2+x1, x7^2-x2, -x7^2+x2]
g0XY = [x6^2-((0.5*tau)^2+x2+tau*x5), -x6^2+((0.5*tau)^2+x2+tau*x5), 
    x8^2-((0.5*tau)^2+x3+tau*x7), -x8^2+((0.5*tau)^2+x3+tau*x7), x5^2-x1, -x5^2+x1, x7^2-x2, -x7^2+x2,
y6^2-((0.5*tau)^2+y2+tau*y5), -y6^2+((0.5*tau)^2+y2+tau*y5), 
    y8^2-((0.5*tau)^2+y3+tau*y7), -y8^2+((0.5*tau)^2+y3+tau*y7), y5^2-y1, -y5^2+y1, y7^2-y2, -y7^2+y2]
g1 = append!(gu,g0) # for condition 1

# polynomial stochastic closure certificate of degree deg
function scc(deg, S, pri)
    # synthesize SCC by using the standard formulation
    # deg: degree of scc template
    model = Model(optimizer_with_attributes(Mosek.Optimizer))
    set_optimizer_attribute(model, MOI.Silent(), true)
    
    C_dict = Dict()  # key => (symbolic_poly, numeric_poly)
    for q in keys(S1) # DPA states
        for (lab, qn) in S[q]
            C, Cc, Cb = add_paramC_poly!(model, var, deg, q, qn) # CC, CC coefficients, and CC basis functions
            Cf, Ccf, Cbf = add_paramCf_poly!(model, var, deg, q, qn)
            key_c = (q, qn, lab, "c") 
            key_cf = (q, qn, lab, "cf")
            C_dict[key_c] = (C, Cc, Cb)  # store symbolic, coefficients, basis
            C_dict[key_cf] = (Cf, Ccf, Cbf)

            # cond 1
            var_1 = [x1, x2, x3, x4, x5, x6, x7, x8]
            E_C1 = sum(rho * (C(var=>[x1,x2,x3,x4,x5,x6,x7,x8,dyn(var_1,1),dyn(var_1,2),dyn(var_1,3),x4,x5,x6,x7,x8])) for i in eachindex(U))
            add_psatz!(model, -E_C1 + gamma, append!(var_1,[u1,u2,u3,u4,u5]), append!(g[lab],g1), [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 

            for u in U
                gu_gen = [x4^2-((0.5*tau)^2+x1+tau*u), -x4^2+((0.5*tau)^2+x1+tau*u), y4^2-((0.5*tau)^2+y1+tau*u), -y4^2+((0.5*tau)^2+y1+tau*u)]
                E_C2to4 = C(var=>[x1,x2,x3,x4,x5,x6,x7,x8,dyn(var_1,1),dyn(var_1,2),dyn(var_1,3),x4,x5,x6,x7,x8])
                # cond 2
                if pri[q] <= pri[qn]
                    E_Cf2 = Cf(var=>[x1,x2,x3,x4,x5,x6,x7,x8,dyn(var_1,1),dyn(var_1,2),dyn(var_1,3),x4,x5,x6,x7,x8])
                    g2 = append!(gu_gen,g0)
                    add_psatz!(model, -E_Cf2 + gamma - tau3 * (lamda - E_C2to4), var_1, append!(g2,gX), [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 
                end

                for p in keys(S1)
                    # cond 3
                    Cf3e, Ccf3e, Cbf3e = add_paramCf_poly!(model, var, deg, p, qn) # e as in E[.\.]
                    Cf3, Ccf3, Cbf3 = add_paramCf_poly!(model, var, deg, p, q) 
                    key_cf3 = (p, q, "cf") 
                    key_cf3e = (p, qn, "cf")
                    C_dict[key_cf3] = (Cf3, Ccf3, Cbf3)  
                    C_dict[key_cf3e] = (Cf3e, Ccf3e, Cbf3e)
                    var_2 = [y1, y2, y3, y4, y5, y6, y7, y8]
                    g3to5 = append!(gu_gen,g0XY)
                    if pri[p] <= pri[q] && pri[p] <= pri[qn] 
                        E_Cf3 = Cf3e(var=>[x1,x2,x3,x4,x5,x6,x7,x8,dyn(var_2,1),dyn(var_2,2),dyn(var_2,3),y4,y5,y6,y7,y8]) 
                        add_psatz!(model, Cf3 - E_Cf3 - tau3 * (lamda - E_C2to4), var, append!(g3to5,gXY), [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)              
                    end

                    # cond 4
                    C4e, Cc4e, Cb4e = add_paramC_poly!(model, var, deg, p, qn) # e as in E[.\.]
                    C4, Cc4, Cb4 = add_paramC_poly!(model, var, deg, p, q) 
                    key_c4 = (p, q, "c") 
                    key_c4e = (p, qn, "c")
                    C_dict[key_c4] = (C4, Cc4, Cb4)  
                    C_dict[key_c4e] = (C4e, Cc4e, Cb4e)
                    E_C4 = C4e(var=>[x1,x2,x3,x4,x5,x6,x7,x8,dyn(var_2,1),dyn(var_2,2),dyn(var_2,3),y4,y5,y6,y7,y8])
                    add_psatz!(model, C4 - E_C4 - tau3 * (lamda - E_C2to4), var, append!(g3to5,gXY), [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 
    
                    #### non-negativity of C and Cf
                    add_psatz!(model, C4, var, append!(g3to5,gXY), [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true) 
                    add_psatz!(model, Cf3, var, append!(g3to5,gXY), [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                
                    # cond 5
                    for l in odd_pri(pri)
                        V2, Vc2, Vb2 = add_paramV_poly!(model, [x01, x02, x03, x04, x05, x06, x07, x08, y1, y2, y3, y4, y5, y6, y7, y8], deg, l, "q0", q)
                        V1, Vc1, Vb1 = add_paramV_poly!(model, [x01, x02, x03, x04, x05, x06, x07, x08, x1, x2, x3, x4, x5, x6, x7, x8], deg, l, "q0", p)
                        key_v1 = (l, "q0", p, "v")
                        key_v2 = (l, "q0", q, "v")
                        C_dict[key_v1], C_dict[key_v2] = (V1, Vc1, Vb1), (V2, Vc2, Vb2)
                        #### non-negativity of V
                        add_psatz!(model, V1, var_5, append!(gv,g3to5), [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                        add_psatz!(model, V2, var_5, append!(gv,g3to5), [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                
                        C1 = C4(var=>[x01,x02,x03,x04,x05,x06,x07,x08,x1,x2,x3,x4,x5,x6,x7,x8])
                        if l <= pri[p] && pri[p] <= pri[q]
                            add_psatz!(model, -V2 + V1 - delta + tau1*(C1 - lamda) + tau2*(Cf3 - lamda), var_5, append!(gv,g3to5), [], div(deg+sos_tol,2), QUIET=true, CS=false, TS=false, GroebnerBasis=true)
                        end
                    end
                end
            end
        end
    end
        
    optimize!(model) # solve for coefficients
    status = termination_status(model)
    C_eval_dict = Dict() # Get numerical values of coefficients and plug into polynomials
    for (key, (C, Cc, Cb)) in C_dict
        coeff_vals = round.(value.(Cc); digits=error)  # Round each coefficient to 2 decimal places
        C_numeric = sum(coeff_vals[i] * Cb[i] for i in eachindex(Cb))
        C_eval_dict[key] = (C, C_numeric)
    end
    
    # status might be optimal but if all Bc approx 10^{-error}, it's essentially 0 so OPTIMAL != CC.
    p_sat = 1-gamma/lamda # satisfaction probability lower bound
    return status, C_eval_dict, p_sat
end

# the considered DPA 

# A representing GFb
S1 = Dict(
    :"q0" => [("b", :"q1"), ("a", :"q0")],
    :"q1" => [("b", :"q1"), ("a", :"q0")]
)

pri1 = Dict(:"q0" => 1, :"q1" => 2) # priority function

# Simulation
deg = 1 # desired degree of SCC
stats = @timed (status, CC_data, p_sat) = scc(deg, S1, pri1)

println("poly deg: ", deg)
println("status: ", status)
println("sat probability: ", p_sat)
println("Number of SCC polynomials: ", length(CC_data))
println("time: ", stats.time, "\n")

for (k, v) in CC_data
    println(k, ": lambda x1, x2, x3, x4, x5, x6, x7, x8, y1, y2, y3, y4, y5, y6, y7, y8:", v[2], ",")
end

println("Finished")

poly deg: 1
status: OPTIMAL
sat probability: 0.9932
Number of SCC polynomials: 18
time: 19.497996125

("q0", "q1", "b", "cf"): lambda x1, x2, x3, x4, x5, x6, x7, x8, y1, y2, y3, y4, y5, y6, y7, y8:0.71 - 1.19*y8 - 0.68*y7 - 1.31*y6 - 0.56*y5 - 1.23*y4 - 0.6*y3 - 0.66*y2 - 0.63*y1 - 1.3*x3 - 2.18*x2 - 2.21*x1,
("q1", "q0", "a", "c"): lambda x1, x2, x3, x4, x5, x6, x7, x8, y1, y2, y3, y4, y5, y6, y7, y8:0.83 - 0.8*y8 - 1.05*y7 - 0.97*y6 - 0.89*y5 + 0.02*y4 - 0.41*y3 - 0.49*y2 - 0.01*y1 - 0.97*x3 - 1.8*x2 - 2.27*x1,
("q0", "q1", "cf"): lambda x1, x2, x3, x4, x5, x6, x7, x8, y1, y2, y3, y4, y5, y6, y7, y8:12.1 - 0.24*y8 + 0.1*y7 - 0.17*y6 + 0.03*y5 + 0.79*y4 - 0.13*y3 - 0.09*y2 + 0.39*y1 - 0.22*x7 - 0.23*x5 + 0.75*x3 + 0.74*x2 - 0.18*x1,
("q0", "q1", "c"): lambda x1, x2, x3, x4, x5, x6, x7, x8, y1, y2, y3, y4, y5, y6, y7, y8:12.36 - 0.28*y8 + 0.11*y7 - 0.25*y6 + 0.08*y5 + 0.74*y4 - 0.15*y3 - 0.13*y2 + 0.36*y1 - 1.89*x7 - 1.89*x5 + 0.44*x3 + 0.67*x2 + 0.06*x1,
(1, "q0", "q0", "v"): lambda x